Task 1: Data Exploration and Understanding

Initialize Spark Session

In [1]:
# Importing the required libraries from pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hash, rand
from pyspark.sql.functions import count, when

In [2]:
# Initialising spark for distributed data processing with limited memory for efficient execution
spark = SparkSession.builder \
    .appName("HIGGS") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .getOrCreate()
print("Spark Session Created Successfully")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 07:02:22 WARN Utils: Your hostname, Sais-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.223.134.171 instead (on interface en0)
26/06/29 07:02:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 07:02:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully


Load Dataset

In [3]:
# Load transaction dataset from CSV file
df = spark.read.csv("HIGGS.csv.gz", header=False, inferSchema=True)
print("Dataset loaded successfully")

[Stage 1:>                                                          (0 + 1) / 1]

Dataset loaded successfully


Create and assign column names

In [4]:
# One target column label and 28 feature columns from c1 - c28
cols = ["label"] + [f"c{i}" for i in range(1, 29)]

# Assign new column names to dataframe
df = df.toDF(*cols)

Inspect Dataset Schema

In [5]:
# Display schema for structure and data types 
df.printSchema()

root
 |-- label: double (nullable = true)
 |-- c1: double (nullable = true)
 |-- c2: double (nullable = true)
 |-- c3: double (nullable = true)
 |-- c4: double (nullable = true)
 |-- c5: double (nullable = true)
 |-- c6: double (nullable = true)
 |-- c7: double (nullable = true)
 |-- c8: double (nullable = true)
 |-- c9: double (nullable = true)
 |-- c10: double (nullable = true)
 |-- c11: double (nullable = true)
 |-- c12: double (nullable = true)
 |-- c13: double (nullable = true)
 |-- c14: double (nullable = true)
 |-- c15: double (nullable = true)
 |-- c16: double (nullable = true)
 |-- c17: double (nullable = true)
 |-- c18: double (nullable = true)
 |-- c19: double (nullable = true)
 |-- c20: double (nullable = true)
 |-- c21: double (nullable = true)
 |-- c22: double (nullable = true)
 |-- c23: double (nullable = true)
 |-- c24: double (nullable = true)
 |-- c25: double (nullable = true)
 |-- c26: double (nullable = true)
 |-- c27: double (nullable = true)
 |-- c28: double (null

Preview Dataset

In [6]:
# Display the first 20 rows of the dataset
df.limit(20).toPandas()

26/06/29 07:15:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,label,c1,c2,c3,c4,c5,c6,c7,c8,c9,...,c19,c20,c21,c22,c23,c24,c25,c26,c27,c28
0,1.0,0.869293,-0.635082,0.225690,0.327470,-0.689993,0.754202,-0.248573,-1.092064,0.000000,...,-0.010455,-0.045767,3.101961,1.353760,0.979563,0.978076,0.920005,0.721657,0.988751,0.876678
1,1.0,0.907542,0.329147,0.359412,1.497970,-0.313010,1.095531,-0.557525,-1.588230,2.173076,...,-1.138930,-0.000819,0.000000,0.302220,0.833048,0.985700,0.978098,0.779732,0.992356,0.798343
2,1.0,0.798835,1.470639,-1.635975,0.453773,0.425629,1.104875,1.282322,1.381664,0.000000,...,1.128848,0.900461,0.000000,0.909753,1.108330,0.985692,0.951331,0.803252,0.865924,0.780118
3,0.0,1.344385,-0.876626,0.935913,1.992050,0.882454,1.786066,-1.646778,-0.942383,0.000000,...,-0.678379,-1.360356,0.000000,0.946652,1.028704,0.998656,0.728281,0.869200,1.026736,0.957904
4,1.0,1.105009,0.321356,1.522401,0.882808,-1.205349,0.681466,-1.070464,-0.921871,0.000000,...,-0.373566,0.113041,0.000000,0.755856,1.361057,0.986610,0.838085,1.133295,0.872245,0.808487
5,0.0,1.595839,-0.607811,0.007075,1.818450,-0.111906,0.847550,-0.566437,1.581239,2.173076,...,-0.654227,-1.274345,3.101961,0.823761,0.938191,0.971758,0.789176,0.430553,0.961357,0.957818
6,1.0,0.409391,-1.884684,-1.027292,1.672452,-1.604598,1.338015,0.055427,0.013466,2.173076,...,0.069496,1.377130,3.101961,0.869418,1.222083,1.000627,0.545045,0.698653,0.977314,0.828786
7,1.0,0.933895,0.629130,0.527535,0.238033,-0.966569,0.547811,-0.059439,-1.706866,2.173076,...,1.291248,-1.467454,0.000000,0.901837,1.083671,0.979696,0.783300,0.849195,0.894356,0.774879
8,1.0,1.405144,0.536603,0.689554,1.179567,-0.110061,3.202405,-1.526960,-1.576033,0.000000,...,-0.151202,1.163489,0.000000,1.667071,4.039273,1.175828,1.045352,1.542972,3.534827,2.740754
9,1.0,1.176566,0.104161,1.397002,0.479721,0.265513,1.135563,1.534831,-0.253291,0.000000,...,0.268541,0.530334,0.000000,0.833175,0.773968,0.985750,1.103696,0.849140,0.937104,0.812364


Verify Dataset File Size

In [25]:
# Calculate total dataset size in MB for Big Data justification
import os

path = "HIGGS.csv.gz"
total_size = os.path.getsize(path)

print("Total size in MB:", total_size / (1024 * 1024))

# Display the total number of rows and columns in the dataset
print("Total no of Rows:", df.count())
print("Total no of Columns:", len(df.columns))

# Display the count of each label
print("Count of each label in the dataset")
df.select("label").groupBy("label").count().show()

Total size in MB: 2685.935838699341


Total no of Rows: 11000000
Total no of Columns: 29
Count of each label in the dataset


[Stage 27:>                                                         (0 + 1) / 1]

+-----+-------+
|label|  count|
+-----+-------+
|  1.0|5829123|
|  0.0|5170877|
+-----+-------+



Task 2: Data Preprocessing and Feature Engineering

Check for missing values

In [8]:
# Check if there are any missing values in each column
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

[Stage 9:>                                                          (0 + 1) / 1]

+-----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
|label| c1| c2| c3| c4| c5| c6| c7| c8| c9|c10|c11|c12|c13|c14|c15|c16|c17|c18|c19|c20|c21|c22|c23|c24|c25|c26|c27|c28|
+-----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
|    0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|
+-----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+



Handling missing values

In [9]:
# No missing values found, so no rows were dropped or values filled

Partition Check

In [7]:
# Check no of partitions for distributed data processing verification
print("Partitions:", df.rdd.getNumPartitions())

Partitions: 1


In [8]:
# Since dataset has only 1 partition indicating it is not currently distributed across multiple nodes repartition is done
df = df.repartition(8)
print("Partitions:", df.rdd.getNumPartitions())

[Stage 3:>                                                          (0 + 1) / 1]

Partitions: 8


In [11]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

Feature Engineering

In [12]:
# Since all columns are of the type double type casting is not required

Feature Selection

In [13]:
# All features are selected for training the model
feature_cols = [f"c{i}" for i in range(1, 29)]

In [14]:
df.show(5)

+-----+------------------+-------------------+-------------------+------------------+-------------------+------------------+--------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+--------------------+-------------------+-----------------+-------------------+--------------------+--------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|label|                c1|                 c2|                 c3|                c4|                 c5|                c6|                  c7|                 c8|                c9|               c10|                c11|                c12|               c13|               c14|                 c15|                c16|              c17|                c18|                 c19|                 c20|              c21|               c22|       

Vector Assembly

In [15]:
# Combine all the features in single vector column
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

Feature Scaling

In [16]:
# Scale features so all values are in same range
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledfeatures",
    withStd=True,
    withMean=False
)

Build Pipeline

In [17]:
# Build full ML pipeline
pipeline = Pipeline(stages=[
    assembler,
    scaler
])

Train and Transform Data

In [18]:
# Train the ML pipeline 
model = pipeline.fit(df)

In [20]:
# Save the pipeline
model.write().overwrite().save("task2_pipeline")

In [21]:
# Transform the data
processed_df = model.transform(df)


In [22]:
# Display final transformed features 
processed_df.select("scaledfeatures").show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|scaledfeatures                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

Save the data 

In [24]:
# Save the processed data as parquet file 
processed_df.write.mode("overwrite").parquet("task2_processed_data")